# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aryan-kumar-shrivastav/ml_workspace_clone/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane:** CTR / Ranking Opportunity Scoring — "which visible pages are under-capturing clicks and should be prioritised for a snippet/title review?"

**Task type:** binary classification (label = `is_declining_label`, 1 when `trend_direction == "down"`) evaluated as a **ranking** problem. The editor works through a priority queue, so Precision@K is the metric that matters — how many of the top-K flagged pages actually need attention.

**Method choice — Random Forest (primary), with Logistic Regression and Decision Tree as comparison points:**

The skill table says: *yes/no with an observed label → Logistic Regression, then Random Forest.* I'm running all three:

- **Logistic Regression** — the simplest learner; if this scores close to Random Forest,   complexity isn't earning anything.
- **Decision Tree (depth 5)** — interpretable; I can print the full tree and read every   split. A useful mid-point between linear and ensemble.
- **Random Forest (200 trees, depth 10)** — the primary pick. I expect moderate feature   interactions here (e.g. the effect of `days_since_last_update` on decline risk is likely   different for a `feedly article` vs a `keyword article` with high search volume).   Gradient Boosting would also be appropriate, but the reference pipeline (`scripts/03_train_model.py`)   uses these three, and matching the reference makes the comparison table audit-friendly.

**The bar to clear:** the Week-4 baseline — a rule that flags pages ranking ≤ 20 with below-tier-mean CTR, scoring pages by how far below their tier's typical CTR they fall. **All models must beat this on Precision@20, Precision@50, and Precision@100 on the same client-holdout split.** Complexity earns its place only if it moves those numbers.


In [1]:
import sys, warnings, subprocess, os
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
from pathlib import Path

# ── Find (or clone) the repo root — works locally, in Colab, and via nbconvert ──
# Walk up from this notebook's location looking for the sentinel file.
# If not found anywhere, clone fresh from GitHub.
def find_repo_root():
    # 1. Walk up from cwd
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    # 2. Explicit sibling of cwd (nbconvert sometimes runs from work/notebooks/)
    for candidate in [Path.cwd() / "ml_workspace_clone",
                      Path.cwd().parent / "ml_workspace_clone",
                      Path.cwd().parent.parent]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    # 3. Nothing found — clone fresh
    print("Repo not found locally — cloning from GitHub...")
    clone_to = Path.cwd() / "ml_workspace_clone"
    if not clone_to.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/aryan-kumar-shrivastav/ml_workspace_clone",
             str(clone_to)],
            check=True,
        )
    return clone_to

ROOT = find_repo_root()
os.chdir(str(ROOT))                        # make all relative paths work
sys.path.insert(0, str(ROOT / "scripts")) # make ml_utils importable

from ml_utils import (
    MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES,
    precision_at_k, PROCESSED_DIR, RAW_PATH,
)

# ── Run pipeline scripts if processed files don't exist yet ──
if not (PROCESSED_DIR / "refresh_feature_vector.csv").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)
if not (PROCESSED_DIR / "baseline_refresh_queue.csv").exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "02_baseline_score.py")], check=True)

df_raw = pd.read_csv(RAW_PATH)
print(f"Repo root      : {ROOT}")
print(f"Raw rows       : {len(df_raw):,}  |  columns: {df_raw.shape[1]}")
print(f"Clients        : {df_raw['client_id'].nunique()}")
print(f"Declining share: {(df_raw['trend_direction']=='down').mean():.1%}")


Repo root      : /home/claude/ml_workspace_clone
Raw rows       : 30,000  |  columns: 44
Clients        : 32
Declining share: 54.2%


## 2. Split design

**Client holdout (group-aware split)**

The reference pipeline in `scripts/03_train_model.py` uses a **client-holdout split**: every page belonging to any one of the 6 held-out clients goes entirely to the test set; the remaining 26 clients' pages go to train. The same logic is reproduced here verbatim so the split used in the notebook is byte-for-byte identical to the one in the pipeline — ensuring the baseline and model numbers are comparable and reproducible.

**Why client holdout instead of a random row split?**

A random row split leaks client-level patterns into the model. Because content items from the same client share writing style, content type mix, and market segment, a row-level split effectively lets the model learn client identity without it being a feature — inflating held-out scores. Client holdout forces the model to generalise to clients it has never seen, which is closer to real deployment: the model gets applied to new clients.

**Leakage check:**
- `trend_direction` and `trend_pct` (the sources of the label) are excluded from features   — confirmed in `ml_utils.py::MODEL_NUMERIC_FEATURES` and `MODEL_CATEGORICAL_FEATURES`.
- `impressions_last_30d`, `impressions_prev_30d` and similar columns are used as features   but they are inputs to the label formula, not equal to it — they are available at   decision time and do not directly reproduce the label without the threshold logic.
- No future-window column is used; all features are trailing-90-day aggregates present   at the time an editor would review the page.


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score)

RANDOM_STATE = 42

# ── 1. build features exactly as the reference pipeline does ──
df = pd.read_csv(PROCESSED_DIR / "refresh_feature_vector.csv")

num_cols = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
cat_cols = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

num_df = (df[num_cols]
          .apply(pd.to_numeric, errors="coerce")
          .replace([np.inf, -np.inf], np.nan)
          .fillna(0))
cat_df = pd.get_dummies(
    df[cat_cols].fillna("unknown").astype(str),
    prefix=cat_cols, dummy_na=False, dtype=float,
)
X = pd.concat([num_df.reset_index(drop=True), cat_df.reset_index(drop=True)], axis=1)
feature_cols = list(X.columns)
y = df["is_declining_label"].astype(int)

# ── 2. client-holdout split (verbatim from scripts/03_train_model.py) ──
all_idx     = np.arange(len(df))
clients     = df["client_id"].fillna("unknown").astype(str)
unique_cl   = clients.unique()
rng         = np.random.default_rng(RANDOM_STATE)
shuffled    = rng.permutation(unique_cl)
n_test_cl   = max(1, int(round(len(shuffled) * 0.2)))
test_cl     = set(shuffled[:n_test_cl])
test_mask   = clients.isin(test_cl).to_numpy()
train_idx   = all_idx[~test_mask]
test_idx    = all_idx[test_mask]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Split strategy : client_holdout")
print(f"Train clients  : {len(unique_cl) - n_test_cl}  |  test clients: {n_test_cl}")
print(f"Train rows     : {len(train_idx):,}  |  test rows: {len(test_idx):,}")
print(f"Label rate     : train {y_train.mean():.3f}  |  test {y_test.mean():.3f}")
print(f"Feature count  : {len(feature_cols)}")
print()
print("Leakage check — banned columns NOT in feature list:")
banned = ["trend_direction", "trend_pct", "is_declining_label"]
found  = [c for c in banned if c in feature_cols]
print(f"  Found in features: {found or 'none — clean'}")


Split strategy : client_holdout
Train clients  : 26  |  test clients: 6
Train rows     : 27,675  |  test rows: 2,325
Label rate     : train 0.555  |  test 0.391
Feature count  : 52

Leakage check — banned columns NOT in feature list:
  Found in features: none — clean


## 3. Train + compare vs my baseline

Same test set, same metric (Precision@K) as the Week-4 baseline. The baseline score comes from `data/processed/baseline_refresh_queue.csv` produced by `scripts/02_baseline_score.py` — a deterministic rule that flags pages ranking ≤ 20 with below-tier CTR, scoring by the size of the CTR gap.

Metric to beat: **P@50** (the primary metric from Week-2 task framing). All others reported for completeness.


In [3]:
# ── load baseline scores on the same test indices ──
baseline = pd.read_csv(PROCESSED_DIR / "baseline_refresh_queue.csv")
bl_lookup  = baseline.set_index("content_id")["baseline_refresh_score"]
bl_scores  = df.iloc[test_idx]["content_id"].map(bl_lookup).fillna(0).to_numpy()

def metrics_row(label, y_true, scores):
    pred = (scores >= 0.5).astype(int)
    return {
        "model":         label,
        "accuracy":      round(accuracy_score(y_true, pred), 4),
        "precision":     round(precision_score(y_true, pred, zero_division=0), 4),
        "recall":        round(recall_score(y_true, pred, zero_division=0), 4),
        "f1":            round(f1_score(y_true, pred, zero_division=0), 4),
        "roc_auc":       round(roc_auc_score(y_true, scores), 4),
        "avg_precision": round(average_precision_score(y_true, scores), 4),
        "P@20":          round(precision_at_k(y_true, scores, 20), 4),
        "P@50":          round(precision_at_k(y_true, scores, 50), 4),
        "P@100":         round(precision_at_k(y_true, scores, 100), 4),
    }

# ── train models ──
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  LogisticRegression(class_weight="balanced",
                                      max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5,
        min_samples_leaf=50, random_state=RANDOM_STATE,
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10,
        min_samples_leaf=25, n_estimators=200,
        n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

rows = [metrics_row("baseline_rule", y_test, bl_scores)]
trained, probs = {}, {}
for name, m in models.items():
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_test)[:, 1]
    rows.append(metrics_row(name, y_test, prob))
    trained[name] = m
    probs[name]   = prob

comparison = pd.DataFrame(rows).sort_values("P@50", ascending=False).reset_index(drop=True)
base_rate  = float(y_test.mean())

print(f"Test base rate (share declining): {base_rate:.3f}\n")
print(comparison.to_string(index=False))
print()
best = comparison.iloc[0]["model"]
best_p50 = comparison.iloc[0]["P@50"]
bl_p50   = comparison[comparison["model"]=="baseline_rule"]["P@50"].values[0]
print(f"Best model : {best}  |  P@50 = {best_p50:.2f}  (baseline = {bl_p50:.2f})")


Test base rate (share declining): 0.391

              model  accuracy  precision  recall     f1  roc_auc  avg_precision  P@20  P@50  P@100
      random_forest    0.6723     0.5610  0.7437 0.6395   0.7500         0.6182  0.65  0.74   0.72
      decision_tree    0.6766     0.5686  0.7162 0.6339   0.7415         0.5753  0.65  0.66   0.68
logistic_regression    0.6606     0.5659  0.5666 0.5662   0.7003         0.5215  0.35  0.40   0.44
      baseline_rule    0.6086     0.4986  0.1892 0.2743   0.6269         0.4676  0.15  0.24   0.36

Best model : random_forest  |  P@50 = 0.74  (baseline = 0.24)


## 4. Errors and interpretation

**Where is the model wrong?**

*False positives* — pages flagged as declining that aren't — cluster around recently updated `keyword article` pages (all with `days_since_last_update = 20`) that have moderate-to-high impressions and mid-range positions (14–36). The model likely flags them because their volume/position profile looks similar to genuinely declining pages; the freshness signal (`days_since_last_update`) is present but doesn't override the traffic pattern in these cases.

*False negatives* — declining pages the model misses — are almost entirely near-invisible pages: impressions < 5, `avg_position = 0` (no GSC position data). The model can't detect decline on pages with no measurable search footprint, which is a real gap for any signal-based approach.

**What does the model lean on?**

The top features by built-in Gini importance are `days_with_impressions`, `log_impressions_90d`, and `avg_position` — all measuring search footprint size and consistency. `content_age_days` is fourth. Notably, `ctr` ranks lower (9th) despite being central to the Week-4 rule, suggesting the forest finds the *amount* of search exposure more predictive of decline than the *rate* at which that exposure converts to clicks.

**Sanity check on top feature:** `days_with_impressions` is the count of days in 90 with ≥1 impression — a proxy for how consistently a page appears in search at all. A page that shows up on only 20 of 90 days is structurally more at risk of trend instability than one present all 90 days. This makes intuitive sense and shows no sign of leakage — it's a pre-decision-time aggregate, not derived from the label window.


In [4]:
# ── feature importances for the best model (Random Forest) ──
rf = trained["random_forest"]
fi = (pd.Series(rf.feature_importances_, index=feature_cols)
      .sort_values(ascending=False)
      .head(12))
print("Top 12 features (Gini importance, Random Forest):")
print(fi.round(4).to_string())

# ── error cases: false positives and false negatives ──
test_df = df.iloc[test_idx][[
    "content_id", "content_type", "impressions_90d",
    "avg_position", "ctr", "days_since_last_update", "word_count"
]].copy()
test_df["y_true"] = y_test.values
test_df["prob_rf"] = probs["random_forest"]
test_df["pred"]    = (probs["random_forest"] >= 0.5).astype(int)

fp = (test_df[(test_df.y_true == 0) & (test_df.pred == 1)]
      .sort_values("prob_rf", ascending=False).head(3))
fn = (test_df[(test_df.y_true == 1) & (test_df.pred == 0)]
      .sort_values("prob_rf").head(3))

show_cols = ["content_type","impressions_90d","avg_position","ctr",
             "days_since_last_update","prob_rf"]

print("\n=== FALSE POSITIVES — flagged, but page is NOT declining ===")
print(fp[show_cols].to_string(index=False))
print("\nPattern: recently updated keyword articles with mid-range positions and "
      "moderate impressions — traffic profile looks like decline, but freshness "
      "didn't override the signal.")

print("\n=== FALSE NEGATIVES — missed, page IS declining ===")
print(fn[show_cols].to_string(index=False))
print("\nPattern: near-invisible pages (impressions < 5, avg_position = 0 / no GSC data). "
      "A model that can only see search signals cannot detect decline on pages "
      "that barely show up in search at all.")

# ── decision tree printed for interpretability ──
from sklearn.tree import export_text
dt = trained["decision_tree"]
tree_rules = export_text(dt, feature_names=feature_cols, max_depth=3)
print("\n=== Decision Tree (depth 3 preview) ===")
print(tree_rules[:1800])


Top 12 features (Gini importance, Random Forest):
days_with_impressions    0.1350
log_impressions_90d      0.1294
avg_position             0.1092
content_age_days         0.0920
char_count               0.0387
age_tier_365+            0.0368
log_clicks_90d           0.0366
word_count               0.0354
ctr                      0.0352
scroll_rate              0.0339
days_with_sessions       0.0334
log_sessions_90d         0.0270

=== FALSE POSITIVES — flagged, but page is NOT declining ===
   content_type  impressions_90d  avg_position  ctr  days_since_last_update  prob_rf
keyword article             5091          14.1 0.20                      20 0.737130
keyword article             1076          25.6 0.09                      20 0.734944
keyword article             3026          35.9 0.00                      20 0.733631

Pattern: recently updated keyword articles with mid-range positions and moderate impressions — traffic profile looks like decline, but freshness didn't override th

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Model-vs-baseline verdict (P@50, client-holdout test set):**

| Model | P@20 | P@50 | P@100 | ROC-AUC |
|---|---|---|---|---|
| baseline_rule | 0.15 | 0.24 | 0.36 | 0.63 |
| logistic_regression | 0.35 | 0.40 | 0.44 | 0.70 |
| decision_tree | 0.65 | 0.66 | 0.68 | 0.74 |
| **random_forest** | **0.65** | **0.74** | **0.72** | **0.75** |

Random Forest is the clear winner: P@50 = 0.74 vs the baseline's 0.24 — meaning 74% of the top-50 flagged pages are genuinely declining, vs 24% for the rule. This is a 3× improvement at the most decision-relevant cut.

**Honest framing:** these numbers are observed on a 6-client holdout from a 32-client dataset. Six held-out clients is a small test population — the spread of results across different 6-client samples would be noisy. The direction (model >> baseline) is credible; the exact figures should be read as directional, not precise.
